In [1]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

C:\Users\amish\AppData\Local\Temp\ipykernel_29772\3933654057.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader


In [2]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("./pdf")

Found 1 PDF files to process

Processing: Artificial intelligence.pdf
  ✓ Loaded 2 pages

Total documents loaded: 2


In [3]:
### Text splitting get into chunks
### chunk overlap helps in not loosing the context of the search 
def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

chunks=split_documents(all_pdf_documents)
chunks
type(chunks)

Split 2 documents into 7 chunks

Example chunk:
Content: Artificial intelligence (AI) is the capability of computational systems to perform tasks 
typically associated with human intelligence, such as learning, reasoning, problem-
solving, perception, and d...
Metadata: {'producer': 'Microsoft® Word 2016', 'creator': 'Microsoft® Word 2016', 'creationdate': '2026-06-04T12:39:34+05:30', 'author': 'Amish Singh', 'moddate': '2026-06-04T12:39:34+05:30', 'source': 'pdf\\Artificial intelligence.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1', 'source_file': 'Artificial intelligence.pdf', 'file_type': 'pdf'}


list

In [6]:
chunks=split_documents(all_pdf_documents)
chunks
type(chunks)


Split 2 documents into 7 chunks

Example chunk:
Content: Artificial intelligence (AI) is the capability of computational systems to perform tasks 
typically associated with human intelligence, such as learning, reasoning, problem-
solving, perception, and d...
Metadata: {'producer': 'Microsoft® Word 2016', 'creator': 'Microsoft® Word 2016', 'creationdate': '2026-06-04T12:39:34+05:30', 'author': 'Amish Singh', 'moddate': '2026-06-04T12:39:34+05:30', 'source': 'pdf\\Artificial intelligence.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1', 'source_file': 'Artificial intelligence.pdf', 'file_type': 'pdf'}


list

In [4]:
from dotenv import load_dotenv
import tiktoken
# tokens bit pair encoding 
encoder = tiktoken.get_encoding("cl100k_base")


for i, chunk in enumerate(chunks):
    token_ids = encoder.encode(chunk.page_content)

    print(f"\nChunk {i+1}")
    print("Token IDs:", token_ids[:20])  
    print("Number of BPE Tokens:", len(token_ids))


from langchain_google_genai import GoogleGenerativeAIEmbeddings

# Initialize embedding model
embeddings = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-2"
)

# Generate embedding for first chunk
vector = embeddings.embed_query(
    chunks[0].page_content
)


Chunk 1
Token IDs: [9470, 16895, 11478, 320, 15836, 8, 374, 279, 23099, 315, 55580, 6067, 311, 2804, 9256, 720, 87184, 5938, 449, 3823]
Number of BPE Tokens: 184

Chunk 2
Token IDs: [791, 8776, 9021, 315, 15592, 3495, 2997, 6975, 11, 33811, 11, 6677, 720, 84216, 11, 9293, 11, 5933, 4221, 8863]
Number of BPE Tokens: 201

Chunk 3
Token IDs: [20322, 5361, 25492, 315, 54508, 6957, 1202, 3925, 17706, 20, 1483, 21, 60, 8272, 555, 18852, 315, 720, 4338, 52201]
Number of BPE Tokens: 196

Chunk 4
Token IDs: [94917, 720, 791, 4689, 3575, 315, 1675, 15853, 320, 269, 6968, 8, 11478, 706, 1027, 11102, 1139, 720, 2008, 96440]
Number of BPE Tokens: 65

Chunk 5
Token IDs: [42298, 12074, 8040, 26249, 430, 737, 33337, 3094, 14656, 30308, 33811, 430, 12966, 720, 817, 994, 814, 11886, 47623, 477]
Number of BPE Tokens: 178

Chunk 6
Token IDs: [86924, 1990, 1884, 19476, 13, 720, 81434, 13340, 323, 6677, 15009, 58, 868, 60, 2187, 15592, 7620, 311, 4320, 720]
Number of BPE Tokens: 207

Chunk 7
Token IDs: [43

In [6]:
from langchain_milvus import Milvus
from pymilvus import connections

# Create the Milvus vector store
vector_store = Milvus(
    embedding_function=embeddings,
    connection_args={"uri": "http://localhost:19530"},
    collection_name="my_documents",
    drop_old=True
)

connections.connect(alias=vector_store.alias, uri="http://localhost:19530")

vector_store.add_documents(documents=chunks)

results = vector_store.similarity_search_with_score(
    query="what are the goal of ai",
    k=10
)

print("\nTop Matching Chunks:\n")

for i, (doc, score) in enumerate(results, start=1):
    print(f"Chunk {i}")
    print("Similarity Score:", score)
    print(doc.page_content[:200])
    print("-" * 80)

c:\Users\amish\Desktop\docker\.venv\Lib\site-packages\langchain_milvus\vectorstores\milvus.py:408: RuntimeWarning: coroutine 'AsyncMilvusClient._get_connection' was never awaited
  self.drop()
C:\Users\amish\AppData\Local\Temp\ipykernel_29772\659097791.py:12: PyMilvusDeprecationWarning: `connections.connect` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  connections.connect(alias=vector_store.alias, uri="http://localhost:19530")
c:\Users\amish\Desktop\docker\.venv\Lib\site-packages\langchain_milvus\vectorstores\milvus.py:1339: UserWarning: No ids provided and auto_id is False. Setting auto_id to True automatically.
  warnings.warn(
c:\Users\amish\Desktop\docker\.venv\Lib\site-packages\langchain_milvus\vectorstores\milvus.py:580: PyMilvusDeprecationWarning: `Collection` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  self._col_cache = Collection(self.collection_name, using=self.alias)
c:\Use


Top Matching Chunks:

Chunk 1
Similarity Score: 0.4459746181964874
Goals 
The general problem of simulating (or creating) intelligence has been broken into 
subproblems. These consist of particular traits or capabilities that researchers expect an 
intelligent system
--------------------------------------------------------------------------------
Chunk 2
Similarity Score: 0.4862689971923828
The traditional goals of AI research include learning, reasoning, knowledge 
representation, planning, natural language processing, and perception, as well as support 
for robotics.[a] To reach these 
--------------------------------------------------------------------------------
Chunk 3
Similarity Score: 0.4960336685180664
Artificial intelligence (AI) is the capability of computational systems to perform tasks 
typically associated with human intelligence, such as learning, reasoning, problem-
solving, perception, and d
-----------------------------------------------------------------------------

In [8]:
query = "What are the goals of AI?"
results = vector_store.similarity_search_with_score(
    query=query,
    k=10
)

all_chunks = ""

for i, (doc, score) in enumerate(results, start=1):
    print(f"Chunk {i}")
    print("Similarity Score:", score)
    print(doc.page_content[:200])
    print("-" * 80)

    # Concatenate the full chunk
    all_chunks += doc.page_content + "\n"

print("\nConcatenated Chunks:\n")
print(all_chunks)

Chunk 1
Similarity Score: 0.33370721340179443
Goals 
The general problem of simulating (or creating) intelligence has been broken into 
subproblems. These consist of particular traits or capabilities that researchers expect an 
intelligent system
--------------------------------------------------------------------------------
Chunk 2
Similarity Score: 0.3999294638633728
Artificial intelligence (AI) is the capability of computational systems to perform tasks 
typically associated with human intelligence, such as learning, reasoning, problem-
solving, perception, and d
--------------------------------------------------------------------------------
Chunk 3
Similarity Score: 0.4052106738090515
The traditional goals of AI research include learning, reasoning, knowledge 
representation, planning, natural language processing, and perception, as well as support 
for robotics.[a] To reach these 
--------------------------------------------------------------------------------
Chunk 4
Similarity

In [9]:
all_chunks

'Goals \nThe general problem of simulating (or creating) intelligence has been broken into \nsubproblems. These consist of particular traits or capabilities that researchers expect an \nintelligent system to display. The traits described below have received the most attention \nand cover the scope of AI research.[a] \nReasoning and problem-solving\nArtificial intelligence (AI) is the capability of computational systems to perform tasks \ntypically associated with human intelligence, such as learning, reasoning, problem-\nsolving, perception, and decision-making. It is a field of \nresearch in engineering, mathematics and computer science that develops and studies \nmethods and software that enable machines to perceive their environment and \nuse learning and intelligence to take actions that maximize their chances of achieving \ndefined goals.[1] \nHigh-profile applications of AI include advanced web search engines, chatbots, virtual \nassistants, autonomous vehicles, and play and anal

In [10]:
import os 
import dotenv
load_dotenv()
os.environ["GOOGLE_API_KEY"]=os.getenv("GOOGLE_API_KEY")


from langchain.chat_models import init_chat_model

model = init_chat_model(
    "gemini-2.5-flash",
    model_provider="google_genai"
)
model


prompt = f"""
You are a helpful AI assistant.

Answer the user's question ONLY using the context below.
If the answer is not present in the context, say:
"I couldn't find the answer in the provided context."

Context:
{all_chunks}

Question:
{query}

Answer:
"""

response = model.invoke(prompt)
print("AI response:\n")
print(response.content)

AI response:

The traditional goals of AI research include learning, reasoning, knowledge representation, planning, natural language processing, and perception, as well as support for robotics. Additionally, AI aims to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. Some companies also aim to create artificial general intelligence (AGI), which is AI that can complete virtually any cognitive task at least as well as a human.
